# FACULTY SOLUTION — Week 9: The Cordell question — brand equity with a price tag
Computes every answer from the canonical CSVs and asserts it against the golden fixtures.

In [ ]:
import pandas as pd, numpy as np, math, json, itertools
d = {s: pd.read_csv(f'../data/{s}.csv') for s in ['markets', 'rebrand_params', 'nonfuel_states', 'worked_example_markets']}

In [ ]:

def compute(d):
    m = d['markets'].set_index('market'); p = {x.parameter: x.value for x in d['rebrand_params'].itertuples()}
    nf = d['nonfuel_states'].set_index('state')['nonfuel_per_fill']
    r = {'cost_per_site': p['total_cost_musd'] * 1e6 / p['total_sites']}
    pg = pc = 0.0; full = 0.0
    for k in m.index:
        net = m.loc[k, 'halden_benefit_per_fill'] - m.loc[k, 'keep_uplift_per_fill']
        gain = net * p['fills_per_site_year'] * m.loc[k, 'sites'] / 1e6
        cost = m.loc[k, 'sites'] * r['cost_per_site'] / 1e6
        r['net_per_fill_' + k] = net; r['gain_musd_' + k] = gain; r['cost_musd_' + k] = cost
        full += gain
        if net > 0:
            r['payback_years_' + k] = cost / gain; pg += gain; pc += cost
    r['partial_gain_musd'] = pg; r['partial_cost_musd'] = pc; r['partial_payback_years'] = pc / pg
    r['full_net_gain_musd'] = full
    r['pricewar_partial_payback_years'] = pc / (pg * nf['price_war'] / nf['base'])
    return r


In [ ]:
fx = json.load(open('../fixtures/week9_golden.json'))
r = compute(d)
bad = []
for k, v in fx['results'].items():
    a = r[k]
    ok = (str(a) == str(v)) if isinstance(v, str) else math.isclose(float(a), float(v), rel_tol=1e-3, abs_tol=1e-5)
    print(('OK  ' if ok else 'DIFF'), f'{k:36s}', a)
    if not ok: bad.append(k)
assert not bad, bad
print('FACULTY NOTEBOOK REPRODUCES GOLDEN FIXTURES: PASS')